In [ ]:
!pip -q install groq pypdf python-docx

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Groq connection successful"
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
class DocumentVerificationAgent:
    def __init__(self, groq_client):
        self.client = groq_client
        self.status = "INITIALIZED"

    def start(self):
        self.status = "READY"
        print("=" * 50)
        print("   DOCUMENT AUTHENTICITY VERIFICATION AGENT")
        print("=" * 50)
        print("Agent Status : READY")
        print("Waiting for document upload...")
        print("=" * 50)


agent = DocumentVerificationAgent(client)
agent.start()

In [ ]:
from google.colab import files

def ask_for_document():
    print("\n📄 Agent is ready!")
    print("Please upload the document you want to verify.")

    uploaded = files.upload()

    if not uploaded:
        print("❌ No document uploaded.")
        return None

    filename = next(iter(uploaded))
    print(f"\n✅ Document received: {filename}")

    return filename

In [ ]:
def run_agent():
    if agent.status != "READY":
        print("❌ Agent is not ready.")
        return None

    filename = ask_for_document()
    return filename


print("Agent is initialized and ready.")
print("Upload will be requested when run_agent() is called.")

In [ ]:
from pypdf import PdfReader
from docx import Document
import os

def extract_text(filename):
    extension = os.path.splitext(filename)[1].lower()

    if extension == ".pdf":
        reader = PdfReader(filename)
        text = ""

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

        return text.strip()

    elif extension == ".docx":
        doc = Document(filename)
        return "\n".join(
            paragraph.text for paragraph in doc.paragraphs
            if paragraph.text.strip()
        ).strip()

    elif extension == ".txt":
        with open(filename, "r", encoding="utf-8") as file:
            return file.read().strip()

    else:
        raise ValueError(
            "Unsupported file type. Please use PDF, DOCX, or TXT."
        )

In [ ]:
def analyze_document(text):
    if not text.strip():
        return "No readable text was found in the document."

    prompt = f"""
You are a document verification assistant.

Analyze the following document content and identify signs that may indicate
whether the document is genuine, suspicious, or cannot be verified.

Important:
- Do not claim authenticity with certainty based only on text.
- Look for inconsistencies, missing information, unusual wording,
  conflicting dates/names/numbers, and other suspicious patterns.
- Clearly explain your reasoning.
- If there is not enough evidence, say "Unable to verify".

Return your response in this format:

Assessment: [Likely Genuine / Suspicious / Unable to Verify]

Confidence: [Low / Medium / High]

Reasons:
- reason 1
- reason 2
- reason 3

Document Content:
{text}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [ ]:
def start_verification():
    print("\n" + "=" * 55)
    print("       DOCUMENT AUTHENTICITY VERIFICATION AGENT")
    print("=" * 55)
    print("Agent Status: READY ✅")
    print("\nPlease upload your document.")

    filename = ask_for_document()

    if filename is None:
        return

    print("\n🔍 Reading document...")

    try:
        text = extract_text(filename)

        if not text:
            print("❌ No readable text found in the document.")
            return

        print("✅ Document text extracted.")
        print("\n🤖 Analyzing document with AI...")

        result = analyze_document(text)

        print("\n" + "=" * 55)
        print("              VERIFICATION RESULT")
        print("=" * 55)
        print(result)
        print("=" * 55)

    except Exception as e:
        print(f"\n❌ Error: {e}")

In [ ]:
from pdf2image import convert_from_path
import pytesseract

def extract_text(filename):
    extension = os.path.splitext(filename)[1].lower()

    if extension == ".pdf":
        # Try normal PDF text extraction first
        reader = PdfReader(filename)
        text = ""

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

        text = text.strip()

        # If no text was found, use OCR
        if not text:
            print("📷 Scanned PDF detected. Starting OCR...")

            pages = convert_from_path(filename, dpi=200)
            ocr_text = []

            for i, page in enumerate(pages):
                print(f"   Reading page {i + 1}/{len(pages)}...")
                page_text = pytesseract.image_to_string(page)
                ocr_text.append(page_text)

            text = "\n".join(ocr_text).strip()

        return text

    elif extension == ".docx":
        doc = Document(filename)
        return "\n".join(
            paragraph.text for paragraph in doc.paragraphs
            if paragraph.text.strip()
        ).strip()

    elif extension == ".txt":
        with open(filename, "r", encoding="utf-8") as file:
            return file.read().strip()

    else:
        raise ValueError(
            "Unsupported file type. Please use PDF, DOCX, or TXT."
        )

print("✅ OCR-enabled document extractor is ready.")

In [ ]:
import gradio as gr
from PIL import Image
import pytesseract
import os

def verify_document(file):
    if file is None:
        return "Please upload a document."

    filename = file
    ext = os.path.splitext(filename)[1].lower()

    try:
        # Extract text
        if ext == ".pdf":
            reader = PdfReader(filename)
            text = ""

            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text += t + "\n"

            # OCR for scanned PDF
            if not text.strip():
                pages = convert_from_path(filename, dpi=150)

                for page in pages:
                    text += pytesseract.image_to_string(page) + "\n"

        elif ext == ".docx":
            doc = Document(filename)
            text = "\n".join(p.text for p in doc.paragraphs)

        elif ext in [".jpg", ".jpeg", ".png", ".webp"]:
            image = Image.open(filename)
            text = pytesseract.image_to_string(image)

        elif ext == ".txt":
            with open(filename, "r", encoding="utf-8") as f:
                text = f.read()

        else:
            return "Unsupported file. Upload PDF, DOCX, TXT, JPG or PNG."

        if not text.strip():
            return "Could not extract readable information from this document."

        # Groq analysis
        prompt = f"""
You are a document verification AI.

Analyze this document and identify evidence that may indicate
whether it is genuine, suspicious, or cannot be verified.

Do NOT claim that a document is officially authentic unless
there is authoritative verification evidence.

Return EXACTLY this structure:

ASSESSMENT: [Likely Genuine / Suspicious / Unable to Verify]

CONFIDENCE: [Low / Medium / High]

KEY FINDINGS:
1.
2.
3.

EXTRACTED DOCUMENT INFORMATION:
{text[:12000]}
"""

        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )

        result = response.choices[0].message.content

        return f"""
📄 DOCUMENT: {os.path.basename(filename)}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔎 DOCUMENT VERIFICATION RESULT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

{result}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️ AI assessment only — not official proof of authenticity.
"""

    except Exception as e:
        return f"ERROR: {str(e)}"


with gr.Blocks(title="Document Verification Agent") as app:

    gr.Markdown("""
    # 🔎 DOCUMENT AUTHENTICITY VERIFICATION AGENT

    ### 🟢 Agent Status: READY

    Upload a PDF, Word document, image, or TXT file.
    """)

    upload = gr.File(
        label="📤 Upload Document",
        file_types=[".pdf", ".docx", ".txt", ".jpg", ".jpeg", ".png", ".webp"],
        type="filepath"
    )

    button = gr.Button("🔍 VERIFY DOCUMENT")

    output = gr.Textbox(
        label="Verification Result",
        lines=22
    )

    button.click(
        verify_document,
        inputs=upload,
        outputs=output
    )

app.launch(share=True)

In [ ]:
# ============================================================
# FINAL DOCUMENT VERIFICATION AGENT
# Replace the previous Gradio cell with this ONE cell
# ============================================================

import gradio as gr
import os
import re
from PIL import Image
import pytesseract
from pypdf import PdfReader
from docx import Document
from pdf2image import convert_from_path

# ------------------------------------------------------------
# TEXT EXTRACTION
# ------------------------------------------------------------

def extract_document_text(filename):
    ext = os.path.splitext(filename)[1].lower()
    text = ""

    if ext == ".pdf":
        reader = PdfReader(filename)

        for page in reader.pages:
            page_text = page.extract_text() or ""
            text += page_text + "\n"

        # Scanned PDF -> OCR
        if len(text.strip()) < 50:
            text = ""
            pages = convert_from_path(filename, dpi=180)

            for i, page in enumerate(pages):
                print(f"OCR: page {i+1}/{len(pages)}")
                text += pytesseract.image_to_string(page) + "\n"

    elif ext == ".docx":
        doc = Document(filename)

        for p in doc.paragraphs:
            if p.text.strip():
                text += p.text + "\n"

        # Also read tables
        for table in doc.tables:
            for row in table.rows:
                text += " | ".join(cell.text for cell in row.cells) + "\n"

    elif ext in [".jpg", ".jpeg", ".png", ".webp"]:
        image = Image.open(filename)
        text = pytesseract.image_to_string(image)

    elif ext == ".txt":
        with open(filename, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

    else:
        raise ValueError(
            "Unsupported file. Use PDF, DOCX, TXT, JPG, JPEG, PNG or WEBP."
        )

    return text.strip()


# ------------------------------------------------------------
# BASIC DOCUMENT CHECKS
# ------------------------------------------------------------

def document_checks(text):
    checks = []

    if len(text.strip()) < 50:
        checks.append("Very little readable text was extracted.")
    else:
        checks.append(f"Readable text extracted: {len(text):,} characters.")

    # Dates
    dates = re.findall(
        r'\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{1,2}[/-]\d{1,2}[/-]\d{4})\b',
        text
    )

    if dates:
        checks.append(f"Dates detected: {len(dates)}")
    else:
        checks.append("No standard numeric dates detected.")

    # IDs / certificate numbers
    id_patterns = [
        r'\b(?:ID|Id|ID No|Certificate No|Certificate ID|Reference No|Ref No)'
        r'\s*[:#.-]?\s*[A-Za-z0-9/-]{4,}',
        r'\b[A-Z]{2,5}[-/][A-Z0-9]{4,}\b'
    ]

    ids = []

    for pattern in id_patterns:
        ids.extend(re.findall(pattern, text))

    if ids:
        checks.append("Possible document/reference identifier detected.")
    else:
        checks.append("No obvious document/reference identifier detected.")

    # QR-related text
    if re.search(r'\bQR\b|\bQR CODE\b', text, re.IGNORECASE):
        checks.append("QR-code related text detected.")
    else:
        checks.append("No QR-code text detected.")

    return checks


# ------------------------------------------------------------
# AI VERIFICATION
# ------------------------------------------------------------

def ai_verify(text):

    # Keep prompt within a reasonable size
    document_text = text[:14000]

    checks = document_checks(text)
    checks_text = "\n".join("- " + x for x in checks)

    prompt = f"""
You are a DOCUMENT ANALYSIS ASSISTANT.

Your job is NOT to guess whether a document is fake.

Analyze the supplied document evidence conservatively.

IMPORTANT RULES:

1. Do NOT call a document fake merely because it contains:
   - AI terminology
   - marketing language
   - unusual wording
   - words such as "AI", "processor", "magic", "fused", "advanced",
     "smart", "innovative", etc.
2. Such wording is NOT evidence of forgery by itself.
3. Do NOT claim official authenticity unless an authoritative
   verification source has actually been checked.
4. If the evidence is insufficient, use "Unable to Verify".
5. Look for concrete inconsistencies such as:
   - conflicting names
   - impossible or conflicting dates
   - inconsistent certificate/reference numbers
   - missing expected information
   - obvious OCR contradictions
   - duplicated or contradictory fields
6. Separate observations from conclusions.
7. Never invent information that is not present in the document.

Return EXACTLY this format:

ASSESSMENT: [Likely Genuine / Suspicious / Unable to Verify]

CONFIDENCE: [Low / Medium / High]

EVIDENCE:
1. [specific observation]
2. [specific observation]
3. [specific observation]

POSSIBLE ISSUES:
- [issue or "None identified from the available text"]

VERIFICATION LIMITATION:
[Explain whether official external verification was actually performed.
If not, clearly state that this is an AI-assisted document analysis.]

BASIC EXTRACTION CHECKS:
{checks_text}

DOCUMENT:
{document_text}
"""

    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a conservative document-analysis system. "
                        "Never invent verification evidence."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        result = response.choices[0].message.content

        if not result or not result.strip():
            return None

        return result.strip()

    except Exception as e:
        return f"AI ERROR: {str(e)}"


# ------------------------------------------------------------
# MAIN AGENT
# ------------------------------------------------------------

def verify_document(file):

    if file is None:
        return "📤 Please upload a document."

    filename = file if isinstance(file, str) else file.name

    try:
        print(f"\nProcessing: {os.path.basename(filename)}")

        # 1. Extract
        text = extract_document_text(filename)

        if not text:
            return (
                "❌ NO READABLE INFORMATION FOUND\n\n"
                "The document could not be read. "
                "If it is a scanned PDF/image, try a clearer copy."
            )

        # 2. Basic checks
        checks = document_checks(text)

        # 3. AI analysis
        result = ai_verify(text)

        if result is None:
            return (
                f"📄 DOCUMENT: {os.path.basename(filename)}\n\n"
                f"❌ AI analysis returned no result.\n\n"
                f"Readable characters extracted: {len(text):,}\n\n"
                "The document was successfully read, but the AI service "
                "did not return an analysis."
            )

        return f"""
📄 DOCUMENT: {os.path.basename(filename)}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔎 DOCUMENT VERIFICATION RESULT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

{result}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 EXTRACTION INFORMATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Readable characters: {len(text):,}

{chr(10).join("• " + x for x in checks)}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️ IMPORTANT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

This system performs AI-assisted document analysis.
It does NOT prove official authenticity unless an
authoritative verification source is checked.
"""

    except Exception as e:
        return (
            f"❌ PROCESSING ERROR\n\n"
            f"{type(e).__name__}: {str(e)}"
        )


# ------------------------------------------------------------
# GRADIO AGENT UI
# ------------------------------------------------------------

with gr.Blocks(title="Document Authenticity Verification Agent") as app:

    gr.Markdown("""
# 🔎 Document Authenticity Verification Agent

**Upload → Analyze → Review Evidence**

The agent checks document content for concrete inconsistencies
and clearly separates evidence from uncertainty.
""")

    upload = gr.File(
        label="📤 Upload Document",
        file_types=[
            ".pdf",
            ".docx",
            ".txt",
            ".jpg",
            ".jpeg",
            ".png",
            ".webp"
        ],
        type="filepath"
    )

    verify_button = gr.Button(
        "🔍 VERIFY DOCUMENT",
        variant="primary"
    )

    result = gr.Textbox(
        label="Verification Result",
        lines=30,
        show_copy_button=True
    )

    verify_button.click(
        fn=verify_document,
        inputs=upload,
        outputs=result
    )

    gr.Markdown("""
### Supported files
PDF • Word (.docx) • TXT • JPG • JPEG • PNG • WEBP

**Note:** AI analysis alone cannot establish legal or official authenticity.
""")

app.launch(share=True)